# Capítulo 8: Regressão Linear

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 3 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> A visão geral deste capítulo ainda será escrita.

## Seções

| Seção | Tópico |
|---|---|
| [8.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/01-regressao-linear-simples.html) | Regressão Linear Simples |
| [8.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/02-avaliando-o-ajuste.html) | Avaliando o Ajuste: R² e Erro |
| [8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-regressao-multipla.html) | Regressão Múltipla |
| [8.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-preditores-qualitativos.html) | Preditores Qualitativos |
| [8.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/05-interacao-e-termos-nao-lineares.html) | Interação e Termos Não Lineares |
| [8.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-outliers-alavancagem-e-colinearidade.html) | Outliers, Alavancagem e Colinearidade |
| [8.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/07-regressao-linear-contra-k-vizinhos.html) | Regressão Linear contra k-Vizinhos |

## Regressão Linear Simples

> **📌 Nota**
>
> Esta seção corresponde às seções 3.1 e 3.1.1 de James et al. (2023).

`Advertising` traz o quanto duzentos mercados investiram em três mídias de propaganda — televisão, rádio e jornal — e quantas unidades do produto cada um vendeu. A pergunta mais simples que esse dado permite fazer é também a primeira: o investimento em TV, sozinho, ajuda a prever vendas? Regressão linear simples responde ajustando uma reta a exatamente dois números por mercado, `tv` e `vendas`, deixando `radio` e `jornal` de fora por enquanto.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

plt.style.use("estilo-figuras.mplstyle")

### Uma reta para tv e vendas

In [ ]:
propaganda = pd.read_csv("dados/Advertising.csv")
propaganda.shape, propaganda.columns.tolist()

Duzentos mercados, quatro colunas — `tv`, `radio`, `jornal` e `vendas`, a primeira em milhares de dólares, a última em milhares de unidades. Regressão linear simples escreve a relação entre as duas que interessam aqui como

$$
\text{vendas} \approx \beta_0 + \beta_1 \cdot \text{tv}.
$$

$\beta_0$ e $\beta_1$ são duas constantes, desconhecidas até se estimar: $\beta_0$ é o **intercepto** — o valor esperado de vendas quando o investimento em TV é zero —, e $\beta_1$ é a **inclinação** — o quanto vendas muda, em média, para cada unidade a mais de tv. O símbolo "≈" marca que a relação é uma aproximação: nada garante que dois mercados com o mesmo investimento em TV vendam exatamente o mesmo, e o quanto cada um foge da reta é o que o resto desta seção mede.

### O resíduo, e a soma que ele eleva ao quadrado

Uma vez que se tem estimativas $\hat\beta_0$ e $\hat\beta_1$, a reta prevê

$$
\hat y_i = \hat\beta_0 + \hat\beta_1 x_i,
$$

e o **resíduo** daquele mercado é a distância entre o que ele de fato vendeu e o que a reta previu para o mesmo investimento em TV:

$$
e_i = y_i - \hat y_i.
$$

A **soma dos quadrados dos resíduos** (RSS) soma esse erro, ao quadrado, sobre os duzentos mercados:

$$
\text{RSS} = e_1^2 + e_2^2 + \cdots + e_n^2 = \sum_{i=1}^{n} \left(y_i - \hat\beta_0 - \hat\beta_1 x_i\right)^2.
$$

Elevar ao quadrado, em vez de somar o valor absoluto de cada resíduo, pune um erro grande desproporcionalmente mais do que vários erros pequenos, e deixa RSS como uma soma de parábolas em $\beta_0$ e $\beta_1$ — uma superfície com um único fundo, que se acha por fórmula fechada em vez de busca. Ajustar a reta é escolher, entre todos os pares $(\beta_0, \beta_1)$ possíveis, o único que minimiza essa soma: é a esse critério que se dá o nome de **mínimos quadrados**.

> **🔷 Conceito**
>
> O **resíduo** $e_i = y_i - \hat y_i$ mede a distância entre um valor observado e o previsto pela reta. A **soma dos quadrados dos resíduos** (RSS) soma $e_i^2$ sobre todas as observações. **Mínimos quadrados** é o critério que escolhe $\hat\beta_0$ e $\hat\beta_1$ minimizando RSS — nenhum outro par de coeficientes produz uma reta com RSS menor.

### A conta à mão: duas médias bastam

Minimizar RSS por cálculo — derivando em relação a $\beta_0$ e a $\beta_1$ e igualando as duas derivadas a zero — leva a uma fórmula fechada que depende só das médias de `tv` e `vendas`, e dos desvios de cada ponto em relação a elas:

$$
\hat\beta_1 = \frac{\displaystyle\sum_{i=1}^{n} (x_i - \bar x)(y_i - \bar y)}{\displaystyle\sum_{i=1}^{n} (x_i - \bar x)^2}, \qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x.
$$

Não precisa de nenhuma biblioteca de otimização — dá para calcular direto com `numpy`:

In [ ]:
tv = propaganda["tv"]
vendas = propaganda["vendas"]

tv_media = tv.mean()
vendas_media = vendas.mean()
beta1_mao = ((tv - tv_media) * (vendas - vendas_media)).sum() / ((tv - tv_media) ** 2).sum()
beta0_mao = vendas_media - beta1_mao * tv_media

round(tv_media, 2), round(vendas_media, 2), round(beta1_mao, 4), round(beta0_mao, 4), round(beta1_mao * 1000, 1)

O investimento médio em TV é 147,04 (mil dólares); a venda média, 14,02 (mil unidades). A partir só dessas duas médias e dos desvios em relação a elas, $\hat\beta_1$ sai 0,0475 e $\hat\beta_0$, 7,0326. Como `tv` e `vendas` vêm as duas em milhares, $\hat\beta_1 \times 1.000$ traduz a inclinação para a escala do dinheiro gasto: 47,5 — cada mil dólares a mais investidos em TV está associado, em média, a 47,5 unidades a mais vendidas.

### A mesma conta, pronta: `LinearRegression`

O `scikit-learn` resolve a mesma minimização sem passar pelas médias explicitamente. `LinearRegression().fit(X, y)` recebe o preditor e a resposta e devolve um objeto já ajustado, com a inclinação em `.coef_` e o intercepto em `.intercept_` — os dois como array e escalar do `numpy`, por isso o `float(...)` ao redor de cada um daqui em diante. `X` entra como o `DataFrame` que já veio do `pandas`, mesmo sendo de uma coluna só, sem nenhum `.to_numpy()`: o estimador aceita e devolve, em `.feature_names_in_`, o nome de coluna que recebeu.

In [ ]:
X = propaganda[["tv"]]
y = propaganda["vendas"]

modelo = LinearRegression().fit(X, y)
beta1_sklearn = float(modelo.coef_[0])
beta0_sklearn = float(modelo.intercept_)

(
    modelo.feature_names_in_,
    (round(beta0_mao, 4), round(beta1_mao, 4)),
    (round(beta0_sklearn, 4), round(beta1_sklearn, 4)),
    bool(np.allclose([beta0_mao, beta1_mao], [beta0_sklearn, beta1_sklearn])),
)

`feature_names_in_` guarda só `tv`, o único nome que o `DataFrame` de uma coluna carregava. Os dois pares de coeficiente — o calculado à mão e o que saiu do `.fit()` — são (7,0326; 0,0475) nos dois casos, e `np.allclose` confirma: `True`. É a mesma fórmula fechada por trás das duas contas; a segunda só evita escrever as médias à mão.

### A reta, e o resíduo que ela deixa

In [ ]:
# Figura: Duzentos mercados: vendas contra o investimento em TV, com a reta de mínimos quadrados por cima. Cada segmento liga um mercado observado à previsão da reta para o mesmo investimento — o resíduo daquele mercado, o e_i que RSS eleva ao quadrado.
yhat = modelo.predict(X)
grade_tv = np.linspace(tv.min(), tv.max(), 200)
reta_grade = modelo.predict(pd.DataFrame({"tv": grade_tv}))

fig, ax = plt.subplots()
ax.vlines(tv, np.minimum(vendas, yhat), np.maximum(vendas, yhat), color="C1", linewidth=1)
ax.plot(grade_tv, reta_grade, color="C0", linewidth=2, label="reta ajustada")
ax.scatter(tv, vendas, color="C2", s=18, zorder=3, label="observado")
ax.set_xlabel("tv (milhares de dólares)")
ax.set_ylabel("vendas (milhares de unidades)")
ax.legend()
plt.tight_layout()
plt.show()

A reta captura a tendência — vender mais conforme se investe mais em TV —, mas nenhum mercado senta exatamente sobre ela: sempre sobra um segmento. Somar o quadrado dos duzentos segmentos desta figura dá exatamente o RSS que a reta minimiza:

In [ ]:
rss_min = float(((vendas - yhat) ** 2).sum())
round(rss_min, 2)

2.102,53 — nenhuma outra reta, entre todos os pares $(\beta_0, \beta_1)$ possíveis, soma menos que isso.

### Um vale com um fundo só

RSS, como soma de quadrados de uma função linear de $\beta_0$ e $\beta_1$, é uma superfície convexa nesses dois parâmetros: um paraboloide elíptico, sem platôs nem mínimos locais além de um único ponto. Variando $\beta_0$ e $\beta_1$ numa grade ao redor de $(\hat\beta_0, \hat\beta_1)$ e calculando RSS em cada combinação, essa forma aparece em curvas de nível — cada uma liga os pares que produzem o mesmo RSS.

In [ ]:
# Figura: Curvas de nível de RSS sobre (β0, β1), na regressão de vendas sobre tv em Advertising. O ponto marcado é (β̂0, β̂1); cada curva liga pares com o mesmo RSS, e as seis se fecham ao redor desse único ponto — não há outro vale na janela.
grade_beta0 = np.linspace(3.5, 10.5, 300)
grade_beta1 = np.linspace(0.02, 0.075, 300)
malha_beta0, malha_beta1 = np.meshgrid(grade_beta0, grade_beta1)

tv_np = tv.to_numpy()
vendas_np = vendas.to_numpy()
residuo_grade = vendas_np - malha_beta0[:, :, None] - malha_beta1[:, :, None] * tv_np
rss_grade = (residuo_grade ** 2).sum(axis=2)

niveis_rss = np.array([2150.0, 2200.0, 2300.0, 2400.0, 2500.0, 2600.0])

fig, ax = plt.subplots()
contornos = ax.contour(malha_beta0, malha_beta1, rss_grade, levels=niveis_rss, cmap="Blues")
ax.scatter([beta0_sklearn], [beta1_sklearn], color="C1", s=40, zorder=3)
ax.annotate(
    r"$(\hat\beta_0,\ \hat\beta_1)$",
    xy=(beta0_sklearn, beta1_sklearn),
    xytext=(10, -14),
    textcoords="offset points",
    fontsize=9,
)
ax.set_xlabel(r"$\beta_0$")
ax.set_ylabel(r"$\beta_1$")
barra = fig.colorbar(contornos, ax=ax, shrink=0.9, pad=0.02)
barra.set_label("RSS")
plt.tight_layout()
plt.show()

A legenda promete curvas fechadas, então a afirmação se confere contando, não olhando: o próprio objeto que o `contour` devolve guarda, para cada nível, os segmentos de linha que desenhou, e um segmento fecha quando termina no mesmo ponto em que começou.

In [ ]:
segmentos_totais = 0
segmentos_fechados = 0
for segmentos_do_nivel in contornos.allsegs:
    for segmento in segmentos_do_nivel:
        if len(segmento) == 0:
            continue
        segmentos_totais += 1
        if np.allclose(segmento[0], segmento[-1]):
            segmentos_fechados += 1

segmentos_totais, segmentos_fechados, segmentos_fechados == segmentos_totais

Seis níveis, seis segmentos desenhados, e os seis fecham: `segmentos_totais` e `segmentos_fechados` saem iguais, 6 e 6. Nenhuma curva sai cortada pela borda da janela — o que confirma, em vez de supor, que RSS tem um único vale nesta vizinhança de $(\hat\beta_0, \hat\beta_1)$.

In [ ]:
minimo_real_menor_que_grade = bool(rss_min < rss_grade.min())
round(rss_min, 2), round(float(rss_grade.min()), 2), minimo_real_menor_que_grade

O mínimo verdadeiro, 2.102,53, fica abaixo até do menor valor que a própria grade alcança, 2.102,56 — `minimo_real_menor_que_grade` é `True`. Nenhum dos 300×300 pontos testados coincide exatamente com $(\hat\beta_0, \hat\beta_1)$, só passa perto: é a fórmula fechada, não a grade, que encontra o fundo do vale de verdade. RSS diz qual par de coeficientes é o melhor entre os que este dado observou; não diz se essa reta presta para prever vendas em geral, nem quanto de vendas ela de fato explica.

## Avaliando o Ajuste: R² e Erro

> **📌 Nota**
>
> Esta seção corresponde à seção 3.1.3 de James et al. (2023).

Ajustar uma reta por mínimos quadrados sempre dá um $\hat\beta_0$ e um $\hat\beta_1$ — a seção anterior encontrou o par que minimiza RSS para `vendas ~ tv` e não sobrou nenhum outro candidato menor. A pergunta muda agora: o quanto essa reta *serve*. Duas medidas respondem isso, cada uma de um jeito, e as duas partem do mesmo RSS.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use("estilo-figuras.mplstyle")

propaganda = pd.read_csv("dados/Advertising.csv")
X = propaganda[["tv"]]
y = propaganda["vendas"]
modelo = LinearRegression().fit(X, y)
yhat = modelo.predict(X)

### O erro típico de previsão: RSE

O erro-padrão residual (RSE) resume o RSS como um único número, na mesma unidade da resposta. É a raiz do RSS dividido não por $n$, e sim por $n-2$ — a reta já consumiu dois parâmetros, $\hat\beta_0$ e $\hat\beta_1$, antes de sobrar erro para medir:

$$
\text{RSE} = \sqrt{\frac{\text{RSS}}{n-2}}
$$

In [ ]:
n = len(y)
n_menos_2 = n - 2
rss = float(((y - yhat) ** 2).sum())
rse_mao = float(np.sqrt(rss / n_menos_2))
rse_mil_unidades = round(rse_mao * 1000)

mse_sklearn = mean_squared_error(y, yhat)
rse_via_mse = float(np.sqrt(mse_sklearn * n / n_menos_2))

vendas_media = float(y.mean())
percentual_erro = rse_mao / vendas_media * 100

(
    n,
    n_menos_2,
    round(rss, 2),
    round(rse_mao, 4),
    round(rse_via_mse, 4),
    bool(np.isclose(rse_mao, rse_via_mse)),
    rse_mil_unidades,
    round(vendas_media, 4),
    round(percentual_erro, 2),
)

Duzentos mercados, `n_menos_2 = 198` depois de descontar os dois parâmetros que a reta gastou para se ajustar: RSS, 2.102,53, dividido por 198 e com a raiz, dá 3,2587. `mean_squared_error` calcula o mesmo RSS já dividido por $n$ — não por $n-2$ —, então recuperar o RSE a partir dele exige desfazer essa divisão antes de tirar a raiz; feito isso, os dois caminhos batem: 3,2587 nos dois casos, `True`.

Vendas está em milhares de unidades, então 3,2587 quer dizer que a venda observada de um mercado típico se afasta da reta em cerca de 3.259 unidades — para cima ou para baixo, em média. Contra a venda média de 14,0225 mil unidades, esses 3,2587 mil unidades representam 23,24% dela: é esse o tamanho do erro típico de previsão, relativo à própria escala do que se está prevendo.

### R²: a proporção de variância explicada

RSE vem na unidade de `vendas`, e 3,2587 só diz alguma coisa a quem sabe a escala de vendas — não dá para comparar direto com o RSE de um modelo cuja resposta é medida em outra unidade. R² resolve isso descartando a unidade: é a fração do TSS — a variância total da resposta, antes de qualquer reta — que a regressão explica.

$$
\text{TSS} = \sum_{i=1}^n (y_i - \bar y)^2, \qquad R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}
$$

In [ ]:
tss = float(((y - vendas_media) ** 2).sum())
explicada = tss - rss

r2_mao = 1 - rss / tss
r2_sklearn = float(r2_score(y, yhat))
r2_via_score = float(modelo.score(X, y))

(
    round(tss, 2),
    round(explicada, 2),
    round(r2_mao, 4),
    round(r2_sklearn, 4),
    round(r2_via_score, 4),
    bool(np.isclose(r2_mao, r2_sklearn) and np.isclose(r2_mao, r2_via_score)),
    round(r2_mao * 100, 2),
)

TSS, a variância total de vendas antes de qualquer reta, sai 5.417,15; descontado o RSS de 2.102,53, sobram 3.314,62 de variância que a reta explica. A conta à mão, `r2_score` e `.score()` — os três caminhos para o mesmo número — concordam em 0,6119: `True`. Uma reta que usa só o investimento em TV explica 61,19% da variância de vendas, sem que esse número carregue unidade nenhuma — é isso que o torna comparável entre modelos onde o RSE não é.

> **🔷 Conceito**
>
> O **erro-padrão residual** (RSE) é a raiz de $\text{RSS}/(n-2)$: o tamanho típico do resíduo, na unidade da resposta. O **R²** é $1 - \text{RSS}/\text{TSS}$: a fração da variância da resposta que o modelo explica, sempre entre 0 e 1 e sem unidade — o que o torna comparável de um modelo para outro, mesmo quando o RSE não é.

R² alto não garante um modelo bom, nem R² baixo condena um modelo ruim: os dois dependem de quanto do problema é o $\mathrm{Var}(\epsilon)$ que o capítulo anterior chamou de piso irredutível — nenhuma reta, nem a melhor possível, explica a parte da variância que é ruído puro.

### O que a reta previu contra o que cada mercado vendeu

In [ ]:
soma_confere = bool(np.isclose(rss + explicada, tss))
explicada_maior_que_rss = bool(explicada > rss)
soma_confere, explicada_maior_que_rss

RSS mais a parte explicada soma de volta o TSS — `soma_confere` é `True` — e a parte explicada é maior que a não explicada, `explicada_maior_que_rss` também `True`, o mesmo fato que R² > 0,5 já dizia.

In [ ]:
# Figura: À esquerda, previsto contra observado para os duzentos mercados de Advertising, com a diagonal de previsão perfeita: a distância vertical de cada ponto até ela é o resíduo daquele mercado. À direita, o TSS decomposto em RSS (não explicada) e a parte que a reta explica, com R² anotado como a fração de cima.
fig, (ax_diag, ax_barra) = plt.subplots(1, 2, figsize=(9, 4.2))

limite = (min(y.min(), yhat.min()) - 1, max(y.max(), yhat.max()) + 1)
ax_diag.plot(limite, limite, color="C1", linewidth=1.5, label="previsão perfeita")
ax_diag.scatter(y, yhat, color="C0", s=16, alpha=0.7, label="mercado")
ax_diag.set_xlim(limite)
ax_diag.set_ylim(limite)
ax_diag.set_aspect("equal")
ax_diag.set_xlabel("vendas observadas (mil unidades)")
ax_diag.set_ylabel("vendas previstas (mil unidades)")
ax_diag.legend()

ax_barra.bar(0, rss, color="C1", label="RSS (não explicada)")
ax_barra.bar(0, explicada, bottom=rss, color="C0", label="explicada (TSS − RSS)")
ax_barra.annotate(
    f"R² = {r2_mao:.3f}",
    xy=(0, rss + explicada / 2),
    xytext=(0.55, rss + explicada / 2),
    va="center",
    fontsize=9,
    arrowprops={"arrowstyle": "-", "linewidth": 0.8},
)
ax_barra.set_xlim(-0.6, 1.6)
ax_barra.set_xticks([0])
ax_barra.set_xticklabels(["TSS"])
ax_barra.set_ylabel("soma de quadrados")
ax_barra.legend(loc="upper right")

plt.tight_layout()
plt.show()

À esquerda, cada ponto é um mercado: quanto mais perto da diagonal, menor o resíduo que a seção anterior somou ao quadrado para formar o RSS. À direita, a mesma barra que soma TSS — verificado acima — dividida nas duas parcelas que R² compara: o pedaço laranja é o RSS que a reta deixou sem explicar, o azul é a fatia que R² = 0,612 mede como proporção do todo.

## Regressão Múltipla

> **📌 Nota**
>
> Esta seção corresponde à seção 3.2 de James et al. (2023).

`Advertising` traz três mídias, e a seção 8.1 usou só uma. A pergunta mais direta é repetir a mesma reta duas vezes mais — uma para `radio`, outra para `jornal` — e ler os três coeficientes lado a lado. Esta seção começa por essa pergunta, e termina desfazendo a resposta que ela parece dar.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

plt.style.use("estilo-figuras.mplstyle")

propaganda = pd.read_csv("dados/Advertising.csv")
y = propaganda["vendas"]

### Jornal, sozinho, parece importar

In [ ]:
X_jornal = propaganda[["jornal"]]
modelo_jornal = LinearRegression().fit(X_jornal, y)
beta1_jornal_sozinho = float(modelo_jornal.coef_[0])

round(beta1_jornal_sozinho, 4), round(beta1_jornal_sozinho * 1000, 1)

Ajustada sozinha contra `jornal`, a reta de mínimos quadrados dá um coeficiente de 0,0547: cada mil dólares a mais gastos em jornal está associado, em média, a 54,7 unidades a mais vendidas. É um coeficiente positivo, do mesmo tipo que a seção 8.1 encontrou para `tv` — nada nesta regressão isolada distingue jornal das outras duas mídias.

Só que ajustar `jornal` sozinho ignora `tv` e `radio` por completo. Se as três mídias variam juntas de mercado para mercado, o coeficiente de uma pode estar levando crédito por vendas que outra produziu — e a única forma de saber é colocar as três na mesma equação.

### As três mídias na mesma equação

O modelo múltiplo estende a mesma ideia de mínimos quadrados a mais de um preditor: em vez de uma reta, um hiperplano que minimiza a soma dos quadrados dos resíduos sobre `tv`, `radio` e `jornal` ao mesmo tempo.

In [ ]:
X_multiplo = propaganda[["tv", "radio", "jornal"]]
modelo_multiplo = LinearRegression().fit(X_multiplo, y)

coeficientes = pd.Series(modelo_multiplo.coef_, index=modelo_multiplo.feature_names_in_)
coeficientes_mil_dolares = (coeficientes * 1000).round(1)

coeficientes.round(4), coeficientes_mil_dolares

`feature_names_in_` guarda os três nomes de coluna que o `DataFrame` carregava, na mesma ordem dos coeficientes — por isso dá para juntar os dois numa única `Series` em vez de decorar qual número é qual. `tv` sai com 0,0458 (45,8 por mil dólares) e `radio` com 0,1885 (188,5 por mil dólares), os dois positivos. `jornal` sai com -0,0010 (-1,0 por mil dólares): praticamente zero, e de sinal oposto ao 0,0547 que a regressão sozinha tinha encontrado para ele.

> **🔷 Conceito**
>
> Num modelo múltiplo, cada coeficiente $\hat\beta_j$ se lê como o efeito médio sobre a resposta de aumentar o preditor $X_j$ em uma unidade, **mantendo todos os outros preditores fixos**. É essa cláusula — mantendo os demais fixos — que separa a leitura de um coeficiente múltiplo da de uma regressão simples, e é ela que muda o que se pode dizer sobre `jornal`.

Lido dessa forma: gastar mais mil dólares em `tv`, mantendo `radio` e `jornal` fixos, está associado a 45,8 unidades a mais de venda; mais mil dólares em `radio`, mantendo `tv` e `jornal` fixos, a 188,5 unidades a mais. Mais mil dólares em `jornal`, mantendo `tv` e `radio` fixos, está associado a uma variação de -1,0 unidade — perto o bastante de zero para não sobrar efeito de jornal depois que as outras duas mídias já estão na conta.

### Por que jornal some: quem anda com quem

A explicação não está em mais uma regressão — está em como as três mídias se relacionam entre si, antes de qualquer venda entrar na conta.

In [ ]:
correlacoes = propaganda[["tv", "radio", "jornal", "vendas"]].corr()
correlacoes.round(4)

In [ ]:
corr_jornal_radio = float(correlacoes.loc["jornal", "radio"])
corr_jornal_tv = float(correlacoes.loc["jornal", "tv"])
jornal_mais_correlacionado_com_radio = bool(corr_jornal_radio > corr_jornal_tv)

round(corr_jornal_radio, 4), round(corr_jornal_tv, 4), jornal_mais_correlacionado_com_radio

`jornal` correlaciona com `radio` a 0,3541 e com `tv` a só 0,0566 — `jornal_mais_correlacionado_com_radio` confirma que a primeira supera a segunda. Mercados que gastam mais em rádio tendem a gastar mais em jornal também: os dois investimentos sobem e descem juntos, sem que isso implique nada causal entre eles.

É essa correlação que explica a reviravolta da seção anterior. Se é o rádio — não o jornal — que de fato move vendas, então nos mercados onde se gasta mais em rádio as vendas tendem a ser maiores, e a tabela de correlação mostra que esses são os mesmos mercados que gastam mais em jornal. Uma regressão que olha só para `jornal`, sem `radio` por perto, não tem como separar as duas coisas: ela atribui a jornal parte do crédito que é do rádio. Colocar as duas mídias na mesma equação é o que permite distinguir — e é aí que o coeficiente de jornal cai a praticamente zero.

### R² e RSE: o quanto o modelo múltiplo melhora

In [ ]:
X_tv = propaganda[["tv"]]
modelo_tv = LinearRegression().fit(X_tv, y)
yhat_tv = modelo_tv.predict(X_tv)

n = len(y)
rss_tv = float(((y - yhat_tv) ** 2).sum())
rse_tv = float(np.sqrt(rss_tv / (n - 2)))
r2_tv = float(r2_score(y, yhat_tv))

round(rse_tv, 4), round(r2_tv, 4)

In [ ]:
yhat_multiplo = modelo_multiplo.predict(X_multiplo)

p = X_multiplo.shape[1]
rss_multiplo = float(((y - yhat_multiplo) ** 2).sum())
rse_multiplo = float(np.sqrt(rss_multiplo / (n - p - 1)))
r2_multiplo = float(r2_score(y, yhat_multiplo))

round(rse_multiplo, 4), round(r2_multiplo, 4)

In [ ]:
comparacao = pd.DataFrame(
    {"R²": [r2_tv, r2_multiplo], "RSE": [rse_tv, rse_multiplo]},
    index=["tv sozinho", "tv + radio + jornal"],
).round(4)
comparacao

O modelo de `tv` sozinho explica 0,6119 da variância de vendas, com erro típico de 3,2587 mil unidades. Somar `radio` e `jornal` sobe o R² para 0,8972 e derruba o RSE para 1,6855: as três mídias juntas explicam mais da variância de vendas do que `tv` sozinho, e erram menos a cada previsão. A régua não muda — R² e RSE continuam sendo as mesmas duas medidas da seção anterior —, só o modelo que elas avaliam.

### A superfície ajustada com tv e radio

Como jornal quase não muda a previsão, a superfície que os coeficientes de `tv` e `radio` desenham já carrega quase toda a informação do modelo múltiplo — e, com só dois preditores, dá para desenhar essa superfície inteira.

In [ ]:
X_tv_radio = propaganda[["tv", "radio"]]
modelo_tv_radio = LinearRegression().fit(X_tv_radio, y)
yhat_tv_radio = modelo_tv_radio.predict(X_tv_radio)

r2_tv_radio = float(r2_score(y, yhat_tv_radio))
round(r2_tv_radio, 4)

In [ ]:
# Figura: Vendas previstas para toda combinação de investimento em tv e radio, pelo modelo ajustado com as duas mídias. Os pontos são os duzentos mercados observados.
grade_tv = np.linspace(propaganda["tv"].min(), propaganda["tv"].max(), 60)
grade_radio = np.linspace(propaganda["radio"].min(), propaganda["radio"].max(), 60)
malha_tv, malha_radio = np.meshgrid(grade_tv, grade_radio)
grade = pd.DataFrame({"tv": malha_tv.ravel(), "radio": malha_radio.ravel()})
superficie = modelo_tv_radio.predict(grade).reshape(malha_tv.shape)

fig, ax = plt.subplots()
mapa = ax.contourf(malha_tv, malha_radio, superficie, levels=14, cmap="Blues")
ax.scatter(propaganda["tv"], propaganda["radio"], color="C1", s=16, edgecolor="white", linewidth=0.5)
ax.set_xlabel("tv (milhares de dólares)")
ax.set_ylabel("radio (milhares de dólares)")
barra = fig.colorbar(mapa, ax=ax, shrink=0.9, pad=0.02)
barra.set_label("vendas prevista (mil unidades)")
plt.tight_layout()
plt.show()

Cada curva de nível reúne as combinações de `tv` e `radio` que o modelo prevê com a mesma venda; a superfície cresce para a direita e para cima, do mesmo jeito que os dois coeficientes positivos calculados acima já anunciavam.

### O que a superfície plana ainda erra

O modelo com `tv` e `radio` explica 0,8972 da variância de vendas — `r2_tv_radio`, medido para este modelo de duas mídias, não o `r2_multiplo` das três —, mas sobra no resíduo um padrão que essa superfície plana não captura.

In [ ]:
df_resid = pd.DataFrame({
    "previsto": yhat_tv_radio,
    "residuo": y.to_numpy() - yhat_tv_radio,
})
df_resid["terco"] = pd.qcut(df_resid["previsto"], 3, labels=["baixo", "medio", "alto"])

medias_por_terco = df_resid.groupby("terco", observed=True)[["previsto", "residuo"]].mean()
medias_por_terco.round(4)

Dividindo os duzentos mercados em três grupos pelo valor previsto — o terço com a previsão mais baixa, o do meio, o mais alto —, o resíduo médio não fica perto de zero nos três grupos: 0,4307 no terço mais baixo, -0,8077 no do meio, e de volta a 0,3650 no mais alto. O sinal do erro típico muda com a faixa de previsão, em vez de se espalhar ao acaso ao redor de zero — um padrão que uma superfície plana, por definição, não reproduz.

In [ ]:
# Figura: Resíduo contra o valor previsto, no modelo de tv e radio. A linha tracejada marca resíduo zero; os três marcadores maiores são a média de cada terço de previsão — abaixo de zero no meio, acima nas duas pontas.
fig, ax = plt.subplots()
ax.axhline(0, color="C2", linewidth=1, linestyle="--")
ax.scatter(df_resid["previsto"], df_resid["residuo"], color="C0", s=14, alpha=0.6)
ax.plot(
    medias_por_terco["previsto"],
    medias_por_terco["residuo"],
    color="C1",
    linewidth=2,
    marker="o",
    markersize=7,
)
ax.set_xlabel("vendas prevista (mil unidades)")
ax.set_ylabel("resíduo (vendas observada − prevista)")
plt.tight_layout()
plt.show()

Essa curvatura — negativa no meio da faixa de previsão, positiva nas duas pontas — é o sinal de que `tv` e `radio` não agem de forma independente sobre vendas. A seção 8.5 retoma este mesmo modelo para mostrar o que muda ao deixar as duas mídias interagirem.

## Preditores Qualitativos

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.1 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Interação e Termos Não Lineares

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.2 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Outliers, Alavancagem e Colinearidade

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.3 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Regressão Linear contra k-Vizinhos

> **📌 Nota**
>
> Esta seção corresponde à seção 3.5 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Leituras adicionais

*A escrever.*

## Referências

- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.